# Activation-Distribution-Aware LoRA Merging: Experimental Research

**Project:** Activation-Distribution-Aware LoRA Merging  
**Base Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Tasks:** BoolQ (QA), GSM8K (Math Reasoning), ARC-Easy (Science QA)  

This notebook provides the complete end-to-end research experimentation pipeline for evaluating whether transforming LoRA updates by input activation covariance square roots prior to SVD alignment and merging improves multi-task performance over weight-space merging under equal rank constraints.

## 1. Setup Environment and Dependencies

In [ ]:
# Install core dependencies
!pip install -q torch transformers peft datasets accelerate scipy pandas pyyaml tqdm safetensors

import os
import sys
import torch
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Configuration Definition

In [ ]:
CONFIG = {
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "revision": "main",
    "seed": 42,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj"],
    "rank_budgets": [4, 8, 16],
    "reg_alpha": 0.001,
    "calib_samples": 128,
    "output_dir": "./lora_experiment_outputs"
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)

## 3. Mathematical Foundations & Matrix Utilities

In [ ]:
def compute_matrix_square_roots(C: torch.Tensor, alpha_reg: float = 1e-3, eps: float = 1e-10):
    """
    Computes regularized C_lambda, C_lambda^{1/2}, and C_lambda^{-1/2} via symmetric eigendecomposition.
    Note: The objective ||E @ C_lambda^{1/2}||_F^2 = E_x[||E x||_2^2] + lambda * ||E||_F^2 is a regularized
    activation-weighted objective (empirical output error + weight-space Frobenius penalty).
    """
    C_64 = C.to(torch.float64)
    C_sym = 0.5 * (C_64 + C_64.T)
    d_in = C_sym.shape[0]
    
    # Tikhonov regularization
    mean_diag = torch.diag(C_sym).mean().item()
    lam = alpha_reg * max(mean_diag, 1e-8)
    C_reg = C_sym + lam * torch.eye(d_in, dtype=torch.float64, device=C.device)
    
    # Eigendecomposition
    eigenvalues, Q = torch.linalg.eigh(C_reg)
    
    # Numerical safeguard
    min_allowable = eps * eigenvalues.max().item()
    clamped_eigs = torch.clamp(eigenvalues, min=max(min_allowable, 1e-12))
    
    sqrt_eigs = torch.sqrt(clamped_eigs)
    inv_sqrt_eigs = 1.0 / sqrt_eigs
    
    C_sqrt = (Q * sqrt_eigs.unsqueeze(0)) @ Q.T
    C_inv_sqrt = (Q * inv_sqrt_eigs.unsqueeze(0)) @ Q.T
    
    return C_sqrt.to(C.dtype), C_inv_sqrt.to(C.dtype)

# Verification on synthetic matrix
test_mat = torch.randn(64, 64)
test_cov = test_mat.T @ test_mat
c_sqrt, c_inv = compute_matrix_square_roots(test_cov)
identity_recon = c_sqrt @ c_inv
err = torch.norm(identity_recon - torch.eye(64)).item()
print(f"Reconstruction Error ||C^(1/2) @ C^(-1/2) - I||_F: {err:.2e}")

## 4. Adapter Alignment & Merging Procedures

In [ ]:
def merge_knots(task_updates, rank_budget=8, inner_method="task_arithmetic", trim_fraction=0.2):
    """
    Standard KnOTS model merging in weight space.
    task_updates: list of torch.Tensor (d_out, d_in)
    """
    # Horizontally concatenate: [dW_1, dW_2, ..., dW_K] -> shape (d_out, K * d_in)
    W_cat = torch.cat(task_updates, dim=1).to(torch.float64)
    d_out, total_in = W_cat.shape
    K = len(task_updates)
    d_in = total_in // K
    
    U, S, Vh = torch.linalg.svd(W_cat, full_matrices=False)
    
    R = min(rank_budget, len(S))
    U_R = U[:, :R]  # (d_out, R)
    S_R = S[:R]      # (R,)
    Vh_R = Vh[:R, :] # (R, K * d_in)
    
    # Partition Vh_R for each task: shape (R, d_in)
    task_aligned_coords = [Vh_R[:, i * d_in : (i + 1) * d_in] for i in range(K)]
    
    if inner_method == "task_arithmetic":
        merged_coords = torch.stack(task_aligned_coords, dim=0).mean(dim=0)
    elif inner_method == "ties":
        # TIES in aligned coordinate space
        stacked = torch.stack(task_aligned_coords, dim=0) # (K, R, d_in)
        # Trim
        k_trim = int(trim_fraction * stacked.numel() / K)
        if k_trim > 0:
            flat = stacked.view(K, -1)
            thresholds = torch.kthvalue(torch.abs(flat), k_trim, dim=1).values.unsqueeze(1)
            trimmed = torch.where(torch.abs(flat) >= thresholds, flat, torch.zeros_like(flat)).view_as(stacked)
        else:
            trimmed = stacked
        # Elect sign
        elected_sign = torch.sign(trimmed.sum(dim=0))
        # Disjoint merge
        mask = (torch.sign(trimmed) == elected_sign) & (trimmed != 0)
        counts = mask.sum(dim=0).clamp(min=1)
        merged_coords = (trimmed * mask).sum(dim=0) / counts
    else:
        raise ValueError(f"Unknown inner method {inner_method}")
    
    # Factorization into LoRA B and A: dW_merged = B @ A
    # Scale distributed as B = U_R, A = diag(S_R) @ merged_coords
    B_merged = U_R.to(task_updates[0].dtype) # (d_out, R)
    A_merged = (torch.diag(S_R) @ merged_coords).to(task_updates[0].dtype) # (R, d_in)
    
    return B_merged, A_merged

def merge_activation_aware_knots(task_updates, cov_matrix, rank_budget=8, inner_method="task_arithmetic", reg_alpha=1e-3, trim_fraction=0.2):
    """
    Activation-Distribution-Aware KnOTS merging.
    Transforms task updates into activation-weighted coordinates before KnOTS alignment,
    then recovers exact rank-R LoRA factors.
    """
    C_sqrt, C_inv_sqrt = compute_matrix_square_roots(cov_matrix, alpha_reg=reg_alpha)
    
    # Transform updates: A_k = dW_k @ C_sqrt
    transformed_updates = [dW @ C_sqrt for dW in task_updates]
    
    # KnOTS in transformed space
    B_trans, A_trans = merge_knots(transformed_updates, rank_budget=rank_budget, inner_method=inner_method, trim_fraction=trim_fraction)
    
    # Invert transformation: A_recovered = A_trans @ C_inv_sqrt
    A_merged = A_trans @ C_inv_sqrt
    B_merged = B_trans
    
    return B_merged, A_merged

## 5. Stage E & F: Independent Adapter Training & Provenance Audit

Execute the automated CLI commands for auditing dataset authenticity and training all three task adapters.

In [ ]:
# 1. Run strict provenance and authenticity audit
!python scripts/audit_datasets.py --config configs/experiment.yaml

# 2. Run multi-adapter preflight check
!python scripts/train_all_adapters.py --preflight

# 3. Train all 3 adapters sequentially on GPU
!python scripts/train_all_adapters.py --device cuda

# 4. Audit and verify saved adapter checkpoints
!python scripts/verify_adapters.py


## 5. Next Steps: Full Experimentation Workflow

1. Train LoRA adapters for BoolQ, GSM8K, and ARC-Easy on `Qwen/Qwen2.5-0.5B-Instruct`.
2. Run strict provenance audit.
3. Collect uncentered activation second moments across 128 calibration examples per task.
4. Execute all 8 benchmark conditions across ranks 4, 8, and 16.
5. Evaluate and report exact accuracy, exact match, and degradation metrics.